# LiTS Tumor Detection with MedSAM2
## Enhanced notebook with organized output folders

This notebook demonstrates:
- Loading and visualizing .nii files from the LiTS dataset
- Using MedSAM2 (latest version) for tumor detection
- Saving results in organized folders:
  - `original_slices/`: Original CT slices
  - `masked_slices/`: Slices with red tumor overlay
  - `overlay_slices/`: Slices with colormap overlay
  - `masks/`: Tumor masks only
  - Detection summary report

## Section 1: Setup and Installation

In [ ]:
# Install required packages
!pip install nibabel matplotlib numpy torch torchvision
!pip install opencv-python-headless

print("\n" + "="*70)
print("INSTALLING MEDSAM2")
print("="*70)

# Clone and install MedSAM2
!git clone https://github.com/bowang-lab/MedSAM.git
%cd MedSAM
!pip install -e .
%cd ..

print("\n✓ Installation complete!")

## Section 2: Import Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
import torch
import cv2
from pathlib import Path
from datetime import datetime

# MedSAM2 imports
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

print("All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Section 3: Download MedSAM2 Checkpoint

In [ ]:
# Download MedSAM2 checkpoint
checkpoint_url = "https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt"
checkpoint_path = "sam2_hiera_large.pt"
model_cfg = "sam2_hiera_l.yaml"

if not os.path.exists(checkpoint_path):
    print("Downloading MedSAM2 checkpoint...")
    !wget {checkpoint_url} -O {checkpoint_path}
    print("✓ Checkpoint downloaded!")
else:
    print("✓ Checkpoint already exists!")

## Section 4: Define TumorDetector Class

In [ ]:
class TumorDetector:
    """Main class for tumor detection using MedSAM2"""
    
    def __init__(self, checkpoint_path, model_cfg="sam2_hiera_l.yaml", device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")
        
        # Build MedSAM2 model
        self.model = build_sam2(model_cfg, checkpoint_path, device=self.device)
        self.predictor = SAM2ImagePredictor(self.model)
        print("MedSAM2 model initialized successfully!")
        
    def preprocess_slice(self, slice_data):
        """Preprocess a CT slice for MedSAM2"""
        slice_normalized = (slice_data - slice_data.min()) / (slice_data.max() - slice_data.min() + 1e-8)
        slice_8bit = (slice_normalized * 255).astype(np.uint8)
        slice_rgb = cv2.cvtColor(slice_8bit, cv2.COLOR_GRAY2RGB)
        return slice_rgb
    
    def get_bounding_box_prompt(self, image_shape, center_ratio=0.5, size_ratio=0.6):
        """Generate a bounding box prompt"""
        h, w = image_shape[:2]
        box_w = int(w * size_ratio)
        box_h = int(h * size_ratio)
        
        x1 = int(w * center_ratio - box_w // 2)
        y1 = int(h * center_ratio - box_h // 2)
        x2 = x1 + box_w
        y2 = y1 + box_h
        
        return np.array([x1, y1, x2, y2])
    
    def detect_tumor(self, slice_data):
        """Detect tumor in a single slice"""
        slice_rgb = self.preprocess_slice(slice_data)
        self.predictor.set_image(slice_rgb)
        
        box_prompt = self.get_bounding_box_prompt(slice_rgb.shape)
        
        with torch.no_grad():
            masks, scores, _ = self.predictor.predict(
                box=box_prompt,
                multimask_output=True
            )
        
        best_mask_idx = np.argmax(scores)
        best_mask = masks[best_mask_idx]
        best_score = scores[best_mask_idx]
        
        return best_mask, best_score, slice_rgb

print("✓ TumorDetector class defined!")

## Section 5: Define NIIProcessor Class

In [ ]:
class NIIProcessor:
    """Class for processing NII files with organized output"""
    
    def __init__(self, nii_path, output_dir="results"):
        self.nii_path = nii_path
        self.output_dir = Path(output_dir)
        
        # Create output directories
        self.original_slices_dir = self.output_dir / "original_slices"
        self.masked_slices_dir = self.output_dir / "masked_slices"
        self.overlay_slices_dir = self.output_dir / "overlay_slices"
        self.masks_dir = self.output_dir / "masks"
        
        for dir_path in [self.original_slices_dir, self.masked_slices_dir, 
                         self.overlay_slices_dir, self.masks_dir]:
            dir_path.mkdir(parents=True, exist_ok=True)
        
        print(f"✓ Output directories created at: {self.output_dir}")
        self.load_nii()
    
    def load_nii(self):
        """Load NII file"""
        try:
            nii_img = nib.load(self.nii_path)
            self.nii_data = nii_img.get_fdata()
            print(f"\n✓ NII file loaded successfully!")
            print(f"  Shape: {self.nii_data.shape}")
            print(f"  Data type: {self.nii_data.dtype}")
            print(f"  Value range: [{self.nii_data.min():.2f}, {self.nii_data.max():.2f}]")
        except Exception as e:
            print(f"✗ Error loading NII file: {e}")
            self.nii_data = None
    
    def get_slice_indices(self, num_slices=10, start_ratio=0.3, end_ratio=0.7):
        """Get slice indices for processing"""
        depth = self.nii_data.shape[2]
        start_idx = int(depth * start_ratio)
        end_idx = int(depth * end_ratio)
        return np.linspace(start_idx, end_idx, num_slices, dtype=int)
    
    def save_slice_visualization(self, slice_data, mask, score, slice_idx):
        """Save all visualizations for a single slice"""
        # Save original slice
        plt.figure(figsize=(8, 8))
        plt.imshow(slice_data.T, cmap='gray', origin='lower')
        plt.title(f'Original Slice {slice_idx}')
        plt.axis('off')
        plt.tight_layout()
        plt.savefig(self.original_slices_dir / f"slice_{slice_idx:04d}.png", 
                    dpi=150, bbox_inches='tight')
        plt.close()
        
        # Save mask only
        plt.figure(figsize=(8, 8))
        plt.imshow(mask.T, cmap='jet', origin='lower')
        plt.title(f'Detected Mask (Score: {score:.3f})')
        plt.axis('off')
        plt.tight_layout()
        plt.savefig(self.masks_dir / f"mask_{slice_idx:04d}.png", 
                    dpi=150, bbox_inches='tight')
        plt.close()
        
        # Save masked slice (red overlay)
        plt.figure(figsize=(8, 8))
        plt.imshow(slice_data.T, cmap='gray', origin='lower')
        mask_overlay = np.zeros((*mask.T.shape, 4))
        mask_overlay[mask.T > 0] = [1, 0, 0, 0.5]
        plt.imshow(mask_overlay, origin='lower')
        plt.title(f'Masked Slice {slice_idx} (Score: {score:.3f})')
        plt.axis('off')
        plt.tight_layout()
        plt.savefig(self.masked_slices_dir / f"masked_{slice_idx:04d}.png", 
                    dpi=150, bbox_inches='tight')
        plt.close()
        
        # Save overlay with colormap
        plt.figure(figsize=(8, 8))
        plt.imshow(slice_data.T, cmap='gray', origin='lower')
        plt.imshow(mask.T, cmap='jet', alpha=0.4, origin='lower')
        plt.title(f'Overlay Slice {slice_idx}')
        plt.axis('off')
        plt.tight_layout()
        plt.savefig(self.overlay_slices_dir / f"overlay_{slice_idx:04d}.png", 
                    dpi=150, bbox_inches='tight')
        plt.close()
    
    def create_summary_report(self, results):
        """Create a summary report"""
        report_path = self.output_dir / "detection_report.txt"
        
        with open(report_path, 'w') as f:
            f.write("=" * 70 + "\n")
            f.write("MEDSAM2 TUMOR DETECTION REPORT\n")
            f.write("=" * 70 + "\n\n")
            f.write(f"NII File: {self.nii_path}\n")
            f.write(f"Volume Shape: {self.nii_data.shape}\n")
            f.write(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
            f.write("=" * 70 + "\n")
            f.write("DETECTION RESULTS\n")
            f.write("=" * 70 + "\n\n")
            
            f.write(f"{'Slice':<10} {'Score':<15} {'Tumor Area (pixels)':<20}\n")
            f.write("-" * 70 + "\n")
            
            for slice_idx, score, area in results:
                f.write(f"{slice_idx:<10} {score:<15.4f} {area:<20}\n")
            
            f.write("\n" + "=" * 70 + "\n")
            f.write("STATISTICS\n")
            f.write("=" * 70 + "\n\n")
            
            scores = [r[1] for r in results]
            areas = [r[2] for r in results]
            
            f.write(f"Total slices processed: {len(results)}\n")
            f.write(f"Average confidence score: {np.mean(scores):.4f}\n")
            f.write(f"Average tumor area: {np.mean(areas):.2f} pixels\n")
            f.write(f"Max tumor area: {np.max(areas):.2f} pixels (Slice {results[np.argmax(areas)][0]})\n")
        
        print(f"\n✓ Summary report saved to: {report_path}")

print("✓ NIIProcessor class defined!")

## Section 6: Upload or Specify NII File

In [ ]:
# OPTION 1: Upload file (Google Colab)
# from google.colab import files
# uploaded = files.upload()
# nii_file = list(uploaded.keys())[0]

# OPTION 2: Specify path
nii_file = "path/to/your/volume.nii"  # UPDATE THIS PATH

# OPTION 3: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# nii_file = '/content/drive/MyDrive/path/to/your/file.nii'

print(f"Using file: {nii_file}")

## Section 7: Initialize System

In [ ]:
# Configuration
OUTPUT_DIR = "medsam2_results"
NUM_SLICES = 10  # Number of slices to process

# Initialize processor
processor = NIIProcessor(nii_file, OUTPUT_DIR)

# Initialize detector
detector = TumorDetector(checkpoint_path, model_cfg)

print("\n✓ System initialized and ready!")

## Section 8: Preview Original Slices

In [ ]:
def preview_slices(nii_data, num_slices=6):
    """Preview original slices"""
    depth = nii_data.shape[2]
    slice_indices = np.linspace(depth//4, 3*depth//4, num_slices, dtype=int)
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for idx, slice_idx in enumerate(slice_indices):
        slice_data = nii_data[:, :, slice_idx]
        axes[idx].imshow(slice_data.T, cmap='gray', origin='lower')
        axes[idx].set_title(f'Slice {slice_idx}/{depth}')
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

if processor.nii_data is not None:
    preview_slices(processor.nii_data)

## Section 9: Run Tumor Detection

In [ ]:
print("\n" + "="*70)
print("STARTING TUMOR DETECTION")
print("="*70 + "\n")

# Get slice indices
slice_indices = processor.get_slice_indices(NUM_SLICES)
print(f"Processing {len(slice_indices)} slices...")
print(f"Slice range: {slice_indices[0]} to {slice_indices[-1]}\n")

# Process slices
results = []

for i, slice_idx in enumerate(slice_indices, 1):
    print(f"[{i}/{len(slice_indices)}] Processing slice {slice_idx}...", end=' ')
    
    slice_data = processor.nii_data[:, :, slice_idx]
    
    try:
        mask, score, _ = detector.detect_tumor(slice_data)
        tumor_area = mask.sum()
        
        # Save visualizations
        processor.save_slice_visualization(slice_data, mask, score, slice_idx)
        
        # Store results
        results.append((slice_idx, score, tumor_area))
        
        print(f"✓ Score: {score:.3f}, Area: {tumor_area} pixels")
        
    except Exception as e:
        print(f"✗ Error: {e}")

# Create summary report
processor.create_summary_report(results)

print("\n" + "="*70)
print("PROCESSING COMPLETE")
print("="*70)

## Section 10: Display Sample Results

In [ ]:
def display_sample_results(processor, num_samples=3):
    """Display sample results from saved files"""
    files = sorted(list(processor.masked_slices_dir.glob("*.png")))
    
    if len(files) == 0:
        print("No results to display")
        return
    
    sample_files = files[:min(num_samples, len(files))]
    
    fig, axes = plt.subplots(1, len(sample_files), figsize=(15, 5))
    if len(sample_files) == 1:
        axes = [axes]
    
    for ax, file in zip(axes, sample_files):
        img = plt.imread(file)
        ax.imshow(img)
        ax.set_title(file.stem)
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

display_sample_results(processor)

## Section 11: Results Summary

In [ ]:
print("\n" + "="*70)
print("RESULTS LOCATION")
print("="*70)
print(f"\nAll results saved to: {OUTPUT_DIR}/")
print(f"\n📁 Folder Structure:")
print(f"  ├── original_slices/     - Original CT slices")
print(f"  ├── masked_slices/       - Slices with red tumor overlay")
print(f"  ├── overlay_slices/      - Slices with colormap overlay")
print(f"  ├── masks/               - Tumor masks only")
print(f"  └── detection_report.txt - Summary statistics")

if results:
    scores = [r[1] for r in results]
    areas = [r[2] for r in results]
    
    print(f"\n📊 Quick Statistics:")
    print(f"  Total slices processed: {len(results)}")
    print(f"  Average confidence: {np.mean(scores):.3f}")
    print(f"  Average tumor area: {np.mean(areas):.0f} pixels")
    print(f"  Max tumor area: {np.max(areas):.0f} pixels")

print("\n" + "="*70)

## Usage Instructions

### Quick Start:
1. Run Section 1 to install dependencies
2. Run Section 2 to import libraries
3. Run Section 3 to download MedSAM2 checkpoint
4. Run Sections 4-5 to define classes
5. Update `nii_file` path in Section 6
6. Run Section 7 to initialize
7. Run Section 9 to start detection

### Output Structure:
- **original_slices/**: Clean CT slices without any overlay
- **masked_slices/**: CT slices with red semi-transparent tumor overlay
- **overlay_slices/**: CT slices with colorful jet colormap overlay
- **masks/**: Just the tumor masks in jet colormap
- **detection_report.txt**: Detailed statistics and results

### Customization:
- Change `NUM_SLICES` in Section 7 to process more/fewer slices
- Modify `start_ratio` and `end_ratio` in `get_slice_indices()` to focus on specific regions
- Adjust `size_ratio` in `get_bounding_box_prompt()` to change detection area

### Tips:
- GPU is highly recommended for faster processing
- MedSAM2 generally performs better than MedSAM (original version)
- Check the detection_report.txt for detailed statistics
- Compare different visualizations to understand results better